In [5]:
import sys
import os
sys.path.insert(0, os.path.abspath('../src'))

from utils.spark_session import get_spark

spark = get_spark()

print(spark)

In [6]:
RAW_PATH = '../data/raw/compas-scores-two-years-violent.csv'

df_raw = (spark.read.option('header', True).option('inferSchema', True).csv(RAW_PATH))

df_raw.printSchema()

print(f'Row count: {df_raw.count()}')


root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- first: string (nullable = true)
 |-- last: string (nullable = true)
 |-- compas_screening_date: date (nullable = true)
 |-- sex: string (nullable = true)
 |-- dob: date (nullable = true)
 |-- age: integer (nullable = true)
 |-- age_cat: string (nullable = true)
 |-- race: string (nullable = true)
 |-- juv_fel_count: integer (nullable = true)
 |-- decile_score11: integer (nullable = true)
 |-- juv_misd_count: integer (nullable = true)
 |-- juv_other_count: integer (nullable = true)
 |-- priors_count14: integer (nullable = true)
 |-- days_b_screening_arrest: integer (nullable = true)
 |-- c_jail_in: timestamp (nullable = true)
 |-- c_jail_out: timestamp (nullable = true)
 |-- c_case_number: string (nullable = true)
 |-- c_offense_date: date (nullable = true)
 |-- c_arrest_date: date (nullable = true)
 |-- c_days_from_compas: integer (nullable = true)
 |-- c_charge_degree: string (nullable = true)
 |-- c_char

In [7]:
BRONZE_PATH = '../data/bronze'

(
    df_raw
    .write
    .format('delta')
    .mode('overwrite')
    .save(BRONZE_PATH)
)

print(f'Bronze layer written to {BRONZE_PATH}')

26/04/07 10:18:43 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: id, name, first, last, compas_screening_date, sex, dob, age, age_cat, race, juv_fel_count, decile_score, juv_misd_count, juv_other_count, priors_count, days_b_screening_arrest, c_jail_in, c_jail_out, c_case_number, c_offense_date, c_arrest_date, c_days_from_compas, c_charge_degree, c_charge_desc, is_recid, r_case_number, r_charge_degree, r_days_from_arrest, r_offense_date, r_charge_desc, r_jail_in, r_jail_out, violent_recid, is_violent_recid, vr_case_number, vr_charge_degree, vr_offense_date, vr_charge_desc, type_of_assessment, decile_score, score_text, screening_date, v_type_of_assessment, v_decile_score, v_score_text, v_screening_date, in_custody, out_custody, priors_count, start, end, event, two_year_recid, two_year_recid
 Schema: id, name, first, last, compas_screening_date, sex, dob, age, age_cat, race, juv_fel_count, decile_score11, juv_misd_count, juv_other_count, priors_count14, days_b_

Bronze layer written to ../data/bronze


In [8]:
df_bronze_check = spark.read.format('delta').load(BRONZE_PATH)

print(f'Bronze row count: {df_bronze_check.count()}')

df_bronze_check.show(5, truncate=False)

from delta.tables import DeltaTable
dt = DeltaTable.forPath(spark, BRONZE_PATH)
print('\nDelta table history:')
dt.history().show(truncate=False)

Bronze row count: 4743
+---+------------------+------+-----------+---------------------+----+----------+---+---------------+----------------+-------------+--------------+--------------+---------------+--------------+-----------------------+-------------------+-------------------+-------------+--------------+-------------+------------------+---------------+------------------------------+--------+-------------+---------------+------------------+--------------+---------------------------+---------+----------+-------------+----------------+--------------+----------------+---------------+---------------------------+------------------+--------------+----------+--------------+--------------------+--------------+------------+----------------+----------+-----------+--------------+-----+----+-----+----------------+----------------+
|id |name              |first |last       |compas_screening_date|sex |dob       |age|age_cat        |race            |juv_fel_count|decile_score11|juv_misd_count|juv_